# Matching road segments and grouping roads

For each crash, find out whether its road went from 50 to 30 km/h in December 2023.

**Sources**
- `ongevallen_gefilterd_amsterdam.csv`: crashes in Amsterdam 2016-2025, with `WVK_ID` (road segment) or `JTE_ID` (junction)
- `data/amsterdam_dertig.json`: road segments with old and new road type (`WEGTYPE_OUD`/`WEGTYPE_NIEUW`), using the old NWB numbering of `WVK_ID`
- `data/nwb_wegvakken_amsterdam.geojson`: current NWB road segments in Amsterdam (PDOK)
- `data/nwb_archive/`: older NWB versions (Rijkswaterstaat, 1 January of each year). Segments get new IDs when the road network is edited, so crashes from earlier years use IDs that are no longer in the current NWB.

**Steps**
1. Group the segments in `amsterdam_dertig.json` by road type
2. Match NWB segments by location to the dertig segments
3. Group junctions by the segments that meet there
4. Do steps 2 and 3 for the current and all older NWB versions
5. Second pass for what is still unknown: street names and junction locations
6. Add the group to the crashes
7. Check the crashes that are still unknown
8. Validate with the recorded speed limits

In [1]:
import json
import os
import tempfile
import urllib.request

import pandas as pd
import geopandas as gpd
import shapely

In [2]:
CRASH_FILE = "ongevallen_gefilterd_amsterdam.csv"
DERTIG_FILE = "data/amsterdam_dertig.json"
NWB_FILE = "data/nwb_wegvakken_amsterdam.geojson"
NWB_URL = ("https://api.pdok.nl/rws/nationaal-wegenbestand-wegen/ogc/v1/collections/wegvakken/items"
           "?f=json&limit=1000&gme_naam=Amsterdam")
ARCHIVE_DIR = "data/nwb_archive"
ARCHIVE_URL = "https://downloads.rijkswaterstaatdata.nl/nwb-wegen/geogegevens/shapefile/Nederland_totaal/{date}.zip"
ARCHIVE_DATES = [f"01-01-{year}" for year in range(2026, 2015, -1)]   # newest first
TRANSLATION_FILE = "data/road_id_translation.csv"
OUTPUT_FILE = "data/crashes_amsterdam_classified.csv"

BUFFER_M = 5          # max distance between old and new segment
MIN_OVERLAP = 0.5     # share of the new segment that must lie on old segments
MIN_PURITY = 0.8      # share of that overlap that must belong to one group
STREET_BUFFER_M = 25  # max distance to a dertig segment with the same street name
JUNCTION_BUFFER_M = 10  # max distance between a junction and the dertig segments around it
FALLBACK_SPEED = 70   # unknown crashes at this recorded speed or higher are on roads that were never 50

## 1. Group segments by road type
- **50->30**: GOW50 became GOW30 or ETW30 (changed)
- **stayed 50**: GOW50 stayed GOW50 (control group)
- **always 30**: was already ETW30 or GOW30
- **outside network**: no road type
- **other**: all other combinations

In [3]:
def road_group(old, new):
    """Group a dertig segment by its old and new road type."""
    if old in ("", "BUITEN NETWERK") and new in ("", "BUITEN N"):
        return "outside network"
    if old == "GOW50" and new in ("GOW30", "ETW30"):
        return "50->30"
    if old == new == "GOW50":
        return "stayed 50"
    if old == new and old in ("ETW30", "GOW30"):
        return "always 30"
    return "other"


# Dutch RD coordinates (EPSG:28992), so lengths are in metres
dertig = gpd.read_file(DERTIG_FILE).to_crs(28992)
dertig["WVK_ID"] = dertig["WVK_ID"].astype(str)
dertig["WEGTYPE_OUD"] = dertig["WEGTYPE_OUD"].fillna("")
dertig["WEGTYPE_NIEUW"] = dertig["WEGTYPE_NIEUW"].fillna("")
dertig["GROUP"] = [road_group(o, n) for o, n in zip(dertig["WEGTYPE_OUD"], dertig["WEGTYPE_NIEUW"])]
dertig = dertig.drop_duplicates("WVK_ID")

print(pd.crosstab([dertig["WEGTYPE_OUD"], dertig["WEGTYPE_NIEUW"]], dertig["GROUP"]))

GROUP                         50->30  always 30  other  outside network  \
WEGTYPE_OUD    WEGTYPE_NIEUW                                              
BUITEN NETWERK                     0          0      0             1511   
               ETW30               0          0     75                0   
               GOW30               0          0     13                0   
               GOW50               0          0      1                0   
ETW30                              0          0      3                0   
               ETW30               0      17001      0                0   
               GOW30               0          0     11                0   
ETW60          ETW30               0          0      1                0   
               ETW60               0          0    109                0   
               GOW50               0          0      1                0   
GOW30          GOW30               0        287      0                0   
GOW50          ETW30     

In [4]:
# Road length per group
(dertig.length.groupby(dertig["GROUP"]).sum() / 1000).round(0).rename("km")

GROUP
50->30              360.0
always 30          1371.0
other               303.0
outside network     272.0
stayed 50           277.0
Name: km, dtype: float64

In [5]:
# Segments with an ID from the dertig file
direct = pd.DataFrame({
    "ID": dertig["WVK_ID"],
    "ID_TYPE": "segment",
    "GROUP": dertig["GROUP"],
    "METHOD": "dertig_id",
    "SOURCE": "dertig",
    "DERTIG_WVK_ID": dertig["WVK_ID"],
})

## 2. Match NWB segments by location
A segment that is not in the dertig file gets the group of the dertig segment it lies on: within 5 m, for at least half of its length, with at least 80% of that overlap in one group. Otherwise it is **unknown**.

In [6]:
buffered = dertig[["WVK_ID", "GROUP", "geometry"]].copy()
buffered["geometry"] = buffered.buffer(BUFFER_M)


def match_segments(nwb, known_ids, source):
    """Match the segments of one NWB version that are not in known_ids to the dertig segments."""
    new = nwb[~nwb["wvk_id"].isin(known_ids)]
    new = new.assign(length=new.length)

    # Pieces of new segments that lie within the buffer of a dertig segment
    pieces = gpd.overlay(new[["wvk_id", "length", "geometry"]], buffered,
                         how="intersection", keep_geom_type=False)
    pieces["overlap"] = pieces.length

    # Per new segment: the dertig segment with the most overlap, and how pure the group is
    best = pieces.sort_values("overlap", ascending=False).drop_duplicates("wvk_id").set_index("wvk_id")
    per_group = pieces.groupby(["wvk_id", "GROUP"])["overlap"].sum()
    best["OVERLAP"] = (best["overlap"] / best["length"]).clip(upper=1)
    best["PURITY"] = per_group.groupby(level=0).max() / per_group.groupby(level=0).sum()
    confident = (best["OVERLAP"] >= MIN_OVERLAP) & (best["PURITY"] >= MIN_PURITY)

    return pd.DataFrame({
        "ID": best.index,
        "ID_TYPE": "segment",
        "GROUP": best["GROUP"].where(confident, "unknown").values,
        "METHOD": "nwb_location",
        "SOURCE": source,
        "DERTIG_WVK_ID": best["WVK_ID"].values,
        "OVERLAP": best["OVERLAP"].round(3).values,
        "PURITY": best["PURITY"].round(3).values,
    })

## 3. Group junctions
A junction (`JTE_ID`) gets the group of the segments that start or end there (`jte_id_beg`/`jte_id_end` in the NWB). Residential streets (**always 30**) and **outside network** are ignored, so a junction on a main road takes the main road's group.
- Different groups left (e.g. 50->30 crossing stayed 50): **mixed**
- Only 30 streets and unmatched segments: **unknown**, since the unmatched segment could be a main road

In [7]:
SIDE_GROUPS = {"always 30", "outside network"}


def junction_group(groups):
    """Group a junction by the groups of the segments that meet there."""
    main = set(groups) - SIDE_GROUPS - {"unknown"}
    if len(main) == 1:
        return main.pop()
    if len(main) > 1:
        return "mixed"
    if "unknown" in groups:
        return "unknown"
    if "always 30" in groups:
        return "always 30"
    return "outside network"


def group_junctions(nwb, segment_group, source):
    """Group the junctions of one NWB version, using the groups of its segments."""
    nwb = nwb.assign(GROUP=nwb["wvk_id"].map(segment_group).fillna("unknown"))
    ends = pd.concat([
        nwb[["jte_id_beg", "GROUP"]].rename(columns={"jte_id_beg": "jte_id"}),
        nwb[["jte_id_end", "GROUP"]].rename(columns={"jte_id_end": "jte_id"}),
    ]).dropna(subset=["jte_id"])
    ends["jte_id"] = ends["jte_id"].astype("int64").astype(str)
    connected = ends.groupby("jte_id")["GROUP"].agg(lambda g: sorted(set(g)))

    return pd.DataFrame({
        "ID": connected.index,
        "ID_TYPE": "junction",
        "GROUP": [junction_group(g) for g in connected],
        "METHOD": "junction",
        "SOURCE": source,
        "CONNECTED_GROUPS": [", ".join(g) for g in connected],
    })

## 4. Current and older NWB versions
The current NWB is downloaded from PDOK, the older versions from the Rijkswaterstaat archive (150-300 MB per version, only the Amsterdam segments are kept). Both are downloaded once and then cached.

Versions are handled newest first. A segment ID gets the group from the first version it appears in. A junction gets the group from the newest version where it is not **unknown**, since in an older version the roads around it may have been matched where the current ones were not.

In [8]:
def download_nwb():
    """Download all current NWB segments in Amsterdam, following the API's pages."""
    url = NWB_URL
    features = []
    while url:
        with urllib.request.urlopen(url, timeout=120) as response:
            page = json.load(response)
        features += page["features"]
        url = next((link["href"] for link in page["links"] if link["rel"] == "next"), None)
    with open(NWB_FILE, "w") as f:
        json.dump({"type": "FeatureCollection", "features": features}, f)
    print(f"Downloaded {len(features)} NWB segments to {NWB_FILE}")


def download_nwb_archive(date):
    """Download one NWB version from the archive and keep the segments in Amsterdam."""
    with tempfile.TemporaryDirectory() as tmp:
        zip_path = f"{tmp}/nwb.zip"
        urllib.request.urlretrieve(ARCHIVE_URL.format(date=date), zip_path)
        nwb = gpd.read_file(f"zip://{zip_path}!{date}/Wegvakken/Wegvakken.shp", where="GME_NAAM = 'Amsterdam'")
    nwb.columns = nwb.columns.str.lower()
    columns = ["wvk_id", "jte_id_beg", "jte_id_end", "stt_naam", "wegbehsrt", "geometry"]
    nwb[columns].to_file(f"{ARCHIVE_DIR}/nwb_{date}.geojson", driver="GeoJSON")
    print(f"Downloaded {len(nwb)} NWB segments for {date}")


def load_nwb(path):
    nwb = gpd.read_file(path).to_crs(28992)
    nwb["wvk_id"] = nwb["wvk_id"].astype(str)
    return nwb


if not os.path.exists(NWB_FILE):
    download_nwb()
os.makedirs(ARCHIVE_DIR, exist_ok=True)
for date in ARCHIVE_DATES:
    if not os.path.exists(f"{ARCHIVE_DIR}/nwb_{date}.geojson"):
        download_nwb_archive(date)

versions = {"current": load_nwb(NWB_FILE)}
versions.update({date: load_nwb(f"{ARCHIVE_DIR}/nwb_{date}.geojson") for date in ARCHIVE_DATES})
pd.Series({name: len(nwb) for name, nwb in versions.items()}, name="segments")

current       45870
01-01-2026    39231
01-01-2025    38993
01-01-2024    38294
01-01-2023    37493
01-01-2022    26186
01-01-2021    25913
01-01-2020    24503
01-01-2019    24237
01-01-2018    23520
01-01-2017    23505
01-01-2016    23487
Name: segments, dtype: int64

In [9]:
# Segments: match the IDs that no earlier version has matched yet
segment_tables = [direct]
for name, nwb in versions.items():
    known_ids = pd.concat(segment_tables)["ID"]
    segment_tables.append(match_segments(nwb, known_ids, name))
segment_table = pd.concat(segment_tables, ignore_index=True)
segment_table["GROUP"].value_counts()

GROUP
always 30          25413
unknown            14560
50->30             10347
stayed 50           5032
outside network     2690
other               1688
Name: count, dtype: int64

## 5. Second pass for what is still unknown
### Street names
The location match fails on some main roads, mostly where newer NWB versions draw the two carriageways more than 5 m apart from the dertig segment. A segment that is still unknown gets the group of the dertig segments **with the same street name within 25 m**, if those are all in one group.

In [10]:
# Every segment ID once, with its geometry from the newest version it is in
segments = pd.concat([nwb[["wvk_id", "stt_naam", "geometry"]].assign(SOURCE=name) for name, nwb in versions.items()])
segments = gpd.GeoDataFrame(segments.drop_duplicates("wvk_id"), crs=28992)

named = dertig[["STT_NAAM", "GROUP", "geometry"]].copy()
named["geometry"] = named.buffer(STREET_BUFFER_M)


def match_street_name(segments):
    """Group of the nearby dertig segments with the same street name, if they are all in one group."""
    near = gpd.sjoin(segments[["wvk_id", "stt_naam", "geometry"]], named, predicate="intersects")
    near = near[near["stt_naam"].str.lower() == near["STT_NAAM"].str.lower()]
    groups = near.groupby("wvk_id")["GROUP"].agg(lambda g: sorted(set(g)))
    return groups[groups.map(len) == 1].str[0]


# Check: does the street name give the same group as the location match, where that one is confident?
located = segment_table[(segment_table["METHOD"] == "nwb_location") & (segment_table["GROUP"] != "unknown")]
located = located.set_index("ID")["GROUP"]
check = match_street_name(segments[segments["wvk_id"].isin(located.index)])
print(f"Street name gives a group for {len(check) / len(located):.0%} of the segments matched by location, "
      f"and the same group for {(check == located[check.index]).mean():.0%} of those")
pd.crosstab(located[check.index].rename("location match"), check.rename("street name"))

Street name gives a group for 69% of the segments matched by location, and the same group for 95% of those


street name,50->30,always 30,other,outside network,stayed 50
location match,,,,,
50->30,3983,148,5,9,74
always 30,147,6202,2,7,45
other,1,0,560,0,24
outside network,12,25,8,473,16
stayed 50,79,60,12,6,1788


In [11]:
# Apply to the segments without a known group
known = segment_table.loc[segment_table["GROUP"] != "unknown", "ID"]
by_street = match_street_name(segments[~segments["wvk_id"].isin(known)])
street_table = pd.DataFrame({
    "ID": by_street.index,
    "ID_TYPE": "segment",
    "GROUP": by_street.values,
    "METHOD": "street_name",
    "SOURCE": segments.set_index("wvk_id").loc[by_street.index, "SOURCE"].values,
})
segment_table = pd.concat([segment_table[~segment_table["ID"].isin(street_table["ID"])], street_table],
                          ignore_index=True)
segment_group = segment_table.set_index("ID")["GROUP"]
print(f"{len(street_table)} segments grouped by street name")
street_table["GROUP"].value_counts()

9862 segments grouped by street name


GROUP
50->30             3870
always 30          3426
stayed 50          1872
other               385
outside network     309
Name: count, dtype: int64

### Junctions
Junctions are grouped by their segments (step 3) only now, so they also use the segments grouped by street name. A junction that is still unknown gets the group of the **dertig segments within 10 m of it**, with the same rules as in step 3.

In [12]:
# Junctions: per version, then keep the newest version where the junction is not unknown
junction_table = pd.concat([group_junctions(nwb, segment_group, name) for name, nwb in versions.items()],
                           ignore_index=True)
junction_table = (junction_table.sort_values("GROUP", key=lambda g: g == "unknown", kind="stable")
                  .drop_duplicates("ID").sort_index())
junction_table["GROUP"].value_counts()

GROUP
always 30          13332
50->30              8580
unknown             3484
stayed 50           3424
other               1124
outside network      964
mixed                838
Name: count, dtype: int64

In [13]:
# Location of every junction: the first or last point of the segments that start or end there
points = []
for name, nwb in versions.items():
    for col, index in [("jte_id_beg", 0), ("jte_id_end", -1)]:
        ends = nwb.dropna(subset=[col])
        points.append(gpd.GeoDataFrame({"ID": ends[col].astype("int64").astype(str).values, "SOURCE": name},
                                        geometry=shapely.get_point(ends.geometry.values, index), crs=28992))
points = pd.concat(points)
points = points[points.geometry.notna()].drop_duplicates("ID")


def match_junction_location(points):
    """Group junctions by the dertig segments within JUNCTION_BUFFER_M of them."""
    near = gpd.sjoin(points, dertig[["GROUP", "geometry"]], predicate="dwithin", distance=JUNCTION_BUFFER_M)
    return near.groupby("ID")["GROUP"].agg(lambda g: junction_group(sorted(set(g))))


# Check: does the location give the same group as the segments, where that one is known?
grouped = junction_table[junction_table["GROUP"] != "unknown"].set_index("ID")["GROUP"]
check = match_junction_location(points[points["ID"].isin(grouped.index)])
print(f"Location gives a group for {len(check) / len(grouped):.0%} of the known junctions, "
      f"and the same group for {(check == grouped[check.index]).mean():.0%} of those")

Location gives a group for 84% of the known junctions, and the same group for 95% of those


In [14]:
# Apply to the junctions that are still unknown
unknown_ids = junction_table.loc[junction_table["GROUP"] == "unknown", "ID"]
by_location = match_junction_location(points[points["ID"].isin(unknown_ids)])
by_location = by_location[by_location != "unknown"]
location_table = pd.DataFrame({
    "ID": by_location.index,
    "ID_TYPE": "junction",
    "GROUP": by_location.values,
    "METHOD": "junction_location",
    "SOURCE": points.set_index("ID").loc[by_location.index, "SOURCE"].values,
})
junction_table = pd.concat([junction_table[~junction_table["ID"].isin(location_table["ID"])], location_table],
                           ignore_index=True)
print(f"{len(location_table)} junctions grouped by location")
location_table["GROUP"].value_counts()

1503 junctions grouped by location


GROUP
always 30          823
outside network    260
50->30             258
stayed 50          120
other               25
mixed               17
Name: count, dtype: int64

In [15]:
translation = pd.concat([segment_table, junction_table], ignore_index=True)
translation.to_csv(TRANSLATION_FILE, index=False)
print(f"Saved {len(translation)} IDs to {TRANSLATION_FILE}")
translation.groupby(["ID_TYPE", "METHOD"])["GROUP"].value_counts().unstack(fill_value=0)

Saved 92994 IDs to data/road_id_translation.csv


GROUP                       50->30  always 30  mixed  other  outside network  \
ID_TYPE  METHOD                                                                
junction junction             8580      13332    838   1124              964   
         junction_location     258        823     17     25              260   
segment  dertig_id            3851      17288      0    841             1511   
         nwb_location         6496       8125      0    847             1179   
         street_name          3870       3426      0    385              309   

GROUP                       stayed 50  unknown  
ID_TYPE  METHOD                                 
junction junction                3424     1981  
         junction_location        120        0  
segment  dertig_id               1795        0  
         nwb_location            3237     6216  
         street_name             1872        0

In [16]:
# How many IDs each version added
pd.crosstab(translation["SOURCE"], translation["ID_TYPE"]).reindex(["dertig", *versions])

ID_TYPE,junction,segment
SOURCE,,
dertig,0,25286
current,27628,29188
01-01-2026,1749,2025
01-01-2025,110,1104
01-01-2024,224,455
01-01-2023,567,1139
01-01-2022,308,1390
01-01-2021,72,95
01-01-2020,267,377


## 6. Add the group to the crashes
- `LOCATION_TYPE`: segment or junction
- `ROAD_GROUP`: group of the road (or **unknown**)
- `MATCH_METHOD`: how the group was found (`dertig_id`, `nwb_location`, `junction`, `street_name`, `junction_location`, `speed_fallback` or `none`)
- `speed_changed`: True for 50->30, False for stayed 50, empty for the rest

Crashes that are still unknown but have a recorded speed limit of 70 km/h or more are on roads that were never 50 (mostly highways), so they go to **other**.

In [17]:
crashes = pd.read_csv(CRASH_FILE, dtype=str, keep_default_na=False)
# IDs are stored as "263360031.0" in the file
for col in ["WVK_ID", "JTE_ID"]:
    crashes[col] = crashes[col].str.removesuffix(".0")
crashes["SPEED"] = pd.to_numeric(crashes["MAXSNELHD"], errors="coerce")

segments = translation[translation["ID_TYPE"] == "segment"].set_index("ID")
junctions = translation[translation["ID_TYPE"] == "junction"].set_index("ID")

on_segment = crashes["WVK_ID"] != ""
crashes["LOCATION_TYPE"] = on_segment.map({True: "segment", False: "junction"})
crashes["ROAD_GROUP"] = crashes["WVK_ID"].map(segments["GROUP"]).where(
    on_segment, crashes["JTE_ID"].map(junctions["GROUP"])).fillna("unknown")
crashes["MATCH_METHOD"] = crashes["WVK_ID"].map(segments["METHOD"]).where(
    on_segment, crashes["JTE_ID"].map(junctions["METHOD"])).fillna("none")

fast = (crashes["ROAD_GROUP"] == "unknown") & (crashes["SPEED"] >= FALLBACK_SPEED)
crashes.loc[fast, "ROAD_GROUP"] = "other"
crashes.loc[fast, "MATCH_METHOD"] = "speed_fallback"
print(f"{fast.sum()} unknown crashes at {FALLBACK_SPEED}+ km/h set to other")

crashes["speed_changed"] = crashes["ROAD_GROUP"].map({"50->30": "True", "stayed 50": "False"}).fillna("")

crashes.drop(columns="SPEED").to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(crashes)} crashes to {OUTPUT_FILE}")

156 unknown crashes at 70+ km/h set to other


Saved 63291 crashes to data/crashes_amsterdam_classified.csv


In [18]:
pd.crosstab(crashes["JAAR_VKL"], crashes["ROAD_GROUP"])

ROAD_GROUP,50->30,always 30,mixed,other,outside network,stayed 50,unknown
JAAR_VKL,,,,,,,
2016,2199,2171,136,1535,72,667,31
2017,2211,1928,193,1733,62,736,21
2018,1989,1904,194,2245,84,693,47
2019,1857,1638,206,2574,60,742,65
2020,1429,1091,159,1633,63,593,50
2021,1587,1266,186,1518,56,652,52
2022,1775,1360,187,1649,71,778,159
2023,1783,1560,203,2269,97,773,128
2024,1616,1314,177,2126,109,721,138


## 7. Crashes that are still unknown

In [19]:
unknown = crashes["ROAD_GROUP"] == "unknown"
print(f"{unknown.sum()} of {len(crashes)} crashes unknown ({unknown.mean():.0%})")

# Share of crashes with a known group, per year
((~unknown).groupby(crashes["JAAR_VKL"]).mean() * 100).round(0).rename("% known")

824 of 63291 crashes unknown (1%)


JAAR_VKL
2016    100.0
2017    100.0
2018     99.0
2019     99.0
2020     99.0
2021     99.0
2022     97.0
2023     98.0
2024     98.0
2025     98.0
Name: % known, dtype: float64

In [20]:
# Why they are unknown: ID not in any NWB version (none), or matched but too uncertain
pd.crosstab([crashes.loc[unknown, "LOCATION_TYPE"], crashes.loc[unknown, "MATCH_METHOD"]],
            crashes.loc[unknown, "SPEED"].fillna(0).astype(int).rename("recorded speed (0 = missing)"))

recorded speed (0 = missing)   0   15   30   50  60
LOCATION_TYPE MATCH_METHOD                         
junction      junction          3   1   23   30   1
              none              0   0    0    2   1
segment       none             61   2   45   71   4
              nwb_location    151  10  139  279   1

Do the unknowns change the conclusion? Before/after ratio of the changed roads compared to the control group (before = 2021-2023, after = 2024-2025), with the unknowns left out, without the second-pass matches (street name and junction location), and in the extreme cases where all unknowns are on a changed road or all on a road that stayed 50. Below 1 means fewer crashes on the changed roads than expected from the control group.

In [21]:
def did_ratio(groups):
    """(after/before on 50->30 roads) / (after/before on stayed 50 roads)."""
    counts = pd.crosstab(crashes["JAAR_VKL"], groups)
    before = counts.loc["2021":"2023"].sum()
    after = counts.loc["2024":"2025"].sum()
    return (after["50->30"] / before["50->30"]) / (after["stayed 50"] / before["stayed 50"])


groups = crashes["ROAD_GROUP"]
pd.Series({
    "unknown left out": did_ratio(groups),
    "without second-pass matches": did_ratio(groups.mask(crashes["MATCH_METHOD"].isin(["street_name", "junction_location"]))),
    "all unknown on 50->30 roads": did_ratio(groups.mask(unknown, "50->30")),
    "all unknown on stayed 50 roads": did_ratio(groups.mask(unknown, "stayed 50")),
}, name="ratio").round(2)

unknown left out                  0.95
without second-pass matches       0.96
all unknown on 50->30 roads       0.96
all unknown on stayed 50 roads    0.91
Name: ratio, dtype: float64

## 8. Validation: recorded speed limits
Each crash record has the speed limit the police recorded (`MAXSNELHD`). On roads that went from 50 to 30, it should go from mostly 50 before the change to mostly 30 after. On roads that stayed 50 it should stay 50.

The recorded speed is too noisy to classify roads with (e.g. a quarter of crashes on roads that stayed 50 are recorded as 30 after the change), but it is a good check of the classification above.

In [22]:
crashes["PERIOD"] = crashes["JAAR_VKL"].map(
    lambda y: "before (2021-2023)" if "2021" <= y <= "2023" else ("after (2024-2025)" if y >= "2024" else None))

# Share of crashes with a recorded speed of 30 or 50, per group and period
checked = crashes.dropna(subset=["PERIOD", "SPEED"])
checked = checked[checked["ROAD_GROUP"].isin(["50->30", "stayed 50", "always 30"])]
speed_share = pd.crosstab([checked["ROAD_GROUP"], checked["PERIOD"]], checked["SPEED"], normalize="index")[[30.0, 50.0]]
(speed_share * 100).round(0).rename(columns={30.0: "% recorded 30", 50.0: "% recorded 50"})

SPEED                          % recorded 30  % recorded 50
ROAD_GROUP PERIOD                                          
50->30     after (2024-2025)            87.0           12.0
           before (2021-2023)           13.0           86.0
always 30  after (2024-2025)            85.0           12.0
           before (2021-2023)           60.0           38.0
stayed 50  after (2024-2025)            26.0           70.0
           before (2021-2023)            7.0           88.0